# Phase 5 · Train with eight chunks and mean pooling

Train on the complete LSI training split. Each document contributes up to eight evenly spaced 512-token chunks. The encoder predicts 100 statute probabilities for every chunk. Their mean gives the document probabilities used in **one binary cross-entropy loss per document**.

The fast default trains the top four encoder blocks and classification head for three epochs. Set TRAIN_TOP_LAYERS = None for full fine-tuning. The eight-chunk cap leaves gaps between sampled windows in exceptionally long cases. Sentence boundaries may cross windows.

Add HF_TOKEN to Colab Secrets with notebook access, or enter it in the hidden prompt. Place best_enc.pt, a1_tokenizer.py, and bpe_40k.json directly in MyDrive. Run cells in order. Checkpoints, token caches, results, and submission files are saved under a separate experiment directory.

The previous inference-only mean-pooling score of 0.3970 used a distilled six-layer model. This notebook starts from the 14-layer Phase 1 encoder, so that number is a reference rather than a controlled comparison. Performance must be measured after training.

Model definitions follow the repository's Phase 5 encoder at commit 8714a5d.

In [1]:
%pip -q install huggingface_hub tqdm scikit-learn

In [2]:
import os, sys, json, math, random, re, hashlib, gc, zipfile
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
from huggingface_hub import hf_hub_download
from getpass import getpass

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DRIVE = Path("/content/drive/MyDrive")
CHECKPOINT = DRIVE / "best_enc.pt"
TOKENIZER_JSON = DRIVE / "bpe_40k.json"
TOKENIZER_PY = DRIVE / "a1_tokenizer.py"
OUT_DIR = DRIVE / "phase5_train_8chunks_mean"
LOCAL_DIR = Path("/content/phase5_chunk_work")
DATA_DIR = None  # Optional local Phase_5_LSI directory; otherwise download with HF_TOKEN.

MAX_CHUNKS = 8
CHUNK_LEN = 512
TRAIN_TOP_LAYERS = 4  # None trains every encoder parameter.
EPOCHS = 3
DOC_BATCH_SIZE = 2
GRAD_ACCUM = 8
CHUNK_MICROBATCH = 4
BACKBONE_LR = 5e-5
HEAD_LR = 1e-3
SEED = 42
RESUME = True
WRITE_SUBMISSION = True
N_LABELS = 100
N_HEADS = 12

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)
for path in (CHECKPOINT, TOKENIZER_JSON, TOKENIZER_PY):
    if not path.is_file():
        raise FileNotFoundError(f"Required file missing: {path}")

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN").strip()
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN and DATA_DIR is None:
    HF_TOKEN = getpass("HF token: ").strip()
if DATA_DIR is None and not HF_TOKEN:
    raise ValueError("Provide HF_TOKEN with access to the competition dataset.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if AMP and torch.cuda.is_bf16_supported() else torch.float16
torch.set_float32_matmul_precision("high")
def autocast_context():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE) if AMP else nullcontext()

sys.path.insert(0, str(TOKENIZER_PY.parent))
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
PAD_ID = tokenizer.token_to_id["<pad>"]
assert CHUNK_LEN <= 512 and MAX_CHUNKS >= 1
assert DOC_BATCH_SIZE > 0 and GRAD_ACCUM > 0 and CHUNK_MICROBATCH > 0
print("Ready. Up to", MAX_CHUNKS, "chunks per document.")

Mounted at /content/drive
Ready. Up to 8 chunks per document.


## Download and cache complete documents
Tokenization preserves the entire source before selecting windows. Placeholder collapsing matches the existing Phase 5 preprocessing. Cached arrays are reused when the data, tokenizer, and chunk settings match.

In [3]:
PLACEHOLDERS = {"<SECTION>": "§", "<ACT>": "¶", "<ENTITY>": "@"}
pattern = re.compile("|".join(re.escape(k) for k in PLACEHOLDERS))
def normalize_text(text):
    return pattern.sub(lambda m: PLACEHOLDERS[m.group()], text)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

TOKENIZER_HASH = sha256_file(TOKENIZER_JSON) + sha256_file(TOKENIZER_PY)
def load_rows(split):
    section = "Test" if split == "test" else "Train"
    if DATA_DIR is None:
        path = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            filename=f"Phase_5_LSI/{section}/{split}.json",
            repo_type="dataset", token=HF_TOKEN, local_dir=str(LOCAL_DIR / "data"))
    else:
        path = Path(DATA_DIR) / section / f"{split}.json"
    with open(path, encoding="utf-8") as stream:
        raw = json.load(stream)
    items = raw.items() if isinstance(raw, dict) else enumerate(raw)
    rows = []
    for key, item in items:
        if not isinstance(item, dict):
            raise ValueError(f"Unexpected record in {split}: {key}")
        text = item.get("text") or item.get("document") or ""
        if isinstance(text, list):
            text = " ".join(map(str, text))
        labels = item.get("labels", item.get("label"))
        if split != "test" and labels is None:
            raise ValueError(f"Missing labels for {split} record {key}")
        if labels is not None and not isinstance(labels, (list, tuple)):
            labels = [labels]
        labels = sorted(set(map(int, labels or [])))
        if any(label < 0 or label >= N_LABELS for label in labels):
            raise ValueError(f"Unexpected statute ID in {split}")
        rows.append({"id": str(item.get("id", key)),
                     "text": normalize_text(str(text)), "labels": labels})
    ids = [r["id"] for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate document IDs in {split}")
    if not rows:
        raise ValueError(f"Empty split: {split}")
    return rows, sha256_file(path)

def select_chunks(ids):
    if not ids:
        ids = [int(tokenizer.token_to_id.get("<s>", 2))]
    length = len(ids)
    count = min(MAX_CHUNKS, max(1, math.ceil(length / CHUNK_LEN)))
    starts = [0] if count == 1 else [
        round(i * (length - CHUNK_LEN) / (count - 1)) for i in range(count)]
    return [ids[start:start + CHUNK_LEN] for start in starts]

def prepare_split(split):
    rows, source_hash = load_rows(split)
    spec = {"version": 1, "source": source_hash, "tokenizer": TOKENIZER_HASH,
            "length": CHUNK_LEN, "chunks": MAX_CHUNKS, "placeholders": PLACEHOLDERS}
    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:20]
    cache = CACHE_DIR / f"{split}_{key}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as data:
            arrays = {name: data[name] for name in data.files}
        print(f"{split}: reused token cache")
    else:
        chunks, lengths, offsets = [], [], [0]
        for row in tqdm(rows, desc=f"Tokenizing {split}", unit="document"):
            ids = tokenizer.encode(row["text"], add_special_tokens=True)
            for chunk in select_chunks(ids):
                lengths.append(len(chunk))
                chunks.append(np.asarray(chunk + [PAD_ID] * (CHUNK_LEN - len(chunk)), dtype=np.int32))
            offsets.append(len(chunks))
        arrays = {"tokens": np.asarray(chunks, dtype=np.int32),
                  "lengths": np.asarray(lengths, dtype=np.int32),
                  "offsets": np.asarray(offsets, dtype=np.int64)}
        temp = cache.with_suffix(".tmp.npz")
        np.savez(temp, **arrays)
        os.replace(temp, cache)
    y = np.zeros((len(rows), N_LABELS), dtype=np.float32)
    for i, row in enumerate(rows):
        y[i, row["labels"]] = 1
    ids = [row["id"] for row in rows]
    assert len(arrays["offsets"]) == len(ids) + 1
    print(f"{split}: {len(ids):,} documents, "
          f"{len(arrays['tokens']) / len(ids):.2f} chunks/document")
    return {"ids": ids, "labels": y, "key": key, **arrays}

train_data = prepare_split("train")
dev_data = prepare_split("dev")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Tokenizing train:   0%|          | 0/42750 [00:00<?, ?document/s]

train: 42,750 documents, 2.60 chunks/document


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Tokenizing dev:   0%|          | 0/10181 [00:00<?, ?document/s]

dev: 10,181 documents, 2.84 chunks/document


## Document batches
Only real chunks enter the encoder. Batches group documents with similar chunk counts, and each forward pass trims padding.

In [4]:
class DocumentDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data["ids"])
    def __getitem__(self, index):
        lo, hi = self.data["offsets"][index:index + 2]
        return (self.data["tokens"][lo:hi], self.data["lengths"][lo:hi],
                self.data["labels"][index], index)

def collate_documents(items):
    tokens, lengths, labels, indices = zip(*items)
    owners = np.repeat(np.arange(len(items)), [len(x) for x in tokens])
    return (torch.from_numpy(np.concatenate(tokens)).long(),
            torch.from_numpy(np.concatenate(lengths)).long(),
            torch.from_numpy(owners).long(),
            torch.from_numpy(np.stack(labels)),
            torch.tensor(indices, dtype=torch.long))

class BucketBatches(Sampler):
    def __init__(self, data, batch_size):
        self.counts = np.diff(data["offsets"])
        self.batch_size = batch_size
        self.epoch = 0
    def __len__(self):
        return math.ceil(len(self.counts) / self.batch_size)
    def __iter__(self):
        rng = np.random.default_rng(SEED + self.epoch)
        order = rng.permutation(len(self.counts))
        batches = []
        bucket = self.batch_size * 100
        for start in range(0, len(order), bucket):
            group = order[start:start + bucket]
            group = group[np.argsort(self.counts[group], kind="stable")]
            batches.extend(group[i:i + self.batch_size].tolist()
                           for i in range(0, len(group), self.batch_size))
        rng.shuffle(batches)
        yield from batches

train_sampler = BucketBatches(train_data, DOC_BATCH_SIZE)
train_loader = DataLoader(DocumentDataset(train_data), batch_sampler=train_sampler,
                          collate_fn=collate_documents, num_workers=0, pin_memory=AMP)
def evaluation_loader(data):
    return DataLoader(DocumentDataset(data), batch_size=DOC_BATCH_SIZE,
                      collate_fn=collate_documents, shuffle=False,
                      num_workers=0, pin_memory=AMP)
dev_loader = evaluation_loader(dev_data)

## Encoder and differentiable chunk pooling
The original encoder weights load strictly. Fast scaled dot-product attention preserves the original attention calculation. The default freezes embeddings and the lower layers, which run without storing gradients. All selected chunks participate in the document loss.

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        mask = None if padding_mask is None else (~padding_mask)[:, None, None, :]
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask,
            dropout_p=self.attn_drop.p if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))

class BackboneCfg:
    pass

def backbone_cfg_from_sd(sd, n_heads=12, max_seq_len=512, dropout=0.1,
                         rope_theta=10000.0):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_heads, c.max_seq_len, c.dropout = n_heads, max_seq_len, dropout
    c.rope_theta = rope_theta
    if c.d_model % c.n_heads:
        raise ValueError(f"d_model {c.d_model} not divisible by n_heads {c.n_heads}")
    c.head_dim = c.d_model // c.n_heads
    return c

In [6]:
class ChunkMeanClassifier(nn.Module):
    def __init__(self, cfg, train_top_layers):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, N_LABELS)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.constant_(self.head.bias, -2.0)
        cos, sin = precompute_rope(cfg.head_dim, CHUNK_LEN, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.freeze_count = 0 if train_top_layers is None else max(0, cfg.n_layers - train_top_layers)
        self.partial = train_top_layers is not None
        if self.partial:
            self.tok_emb.requires_grad_(False)
            for block in self.layers[:self.freeze_count]:
                block.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        if self.partial:
            self.drop.eval()
            for block in self.layers[:self.freeze_count]:
                block.eval()
        return self

    def chunk_logits(self, ids):
        padding_mask = ids == PAD_ID
        with torch.no_grad() if self.partial else nullcontext():
            hidden = self.drop(self.tok_emb(ids))
            for block in self.layers[:self.freeze_count]:
                hidden = block(hidden, self.rope_cos, self.rope_sin, padding_mask)
        for block in self.layers[self.freeze_count:]:
            hidden = block(hidden, self.rope_cos, self.rope_sin, padding_mask)
        hidden = self.norm(hidden)
        keep = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * keep).sum(1) / keep.sum(1).clamp_min(1)
        return self.head(F.dropout(pooled, p=self.cfg.dropout, training=self.training))

    def forward(self, tokens, lengths, owners, n_documents):
        sums = torch.zeros(n_documents, N_LABELS, device=DEVICE, dtype=torch.float32)
        for start in range(0, len(tokens), CHUNK_MICROBATCH):
            end = start + CHUNK_MICROBATCH
            width = int(lengths[start:end].max())
            ids = tokens[start:end, :width].to(DEVICE, non_blocking=True)
            owner = owners[start:end].to(DEVICE, non_blocking=True)
            logits = self.chunk_logits(ids)
            sums = sums.index_add(0, owner, torch.sigmoid(logits.float()))
        counts = torch.bincount(owners, minlength=n_documents).to(DEVICE).float()
        return sums / counts[:, None].clamp_min(1)

checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
state = checkpoint.get("model", checkpoint.get("state_dict", checkpoint))
state = {key.removeprefix("module."): value for key, value in state.items()}
if "tok_emb.weight" not in state or state["layers.0.attn.k.weight"].shape[0] != state["tok_emb.weight"].shape[1]:
    raise ValueError("CHECKPOINT must be your Phase 1 masked encoder best_enc.pt.")
bc = backbone_cfg_from_sd(state, n_heads=N_HEADS, max_seq_len=CHUNK_LEN)
model = ChunkMeanClassifier(bc, TRAIN_TOP_LAYERS)
expected = model.state_dict()
loaded = {key: state[key] if key != "head.weight" and key != "head.bias"
          else value for key, value in expected.items()}
model.load_state_dict(loaded, strict=True)
del checkpoint, state, loaded, expected
gc.collect()
model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Encoder loaded: {bc.n_layers} layers. Trainable parameters: {trainable:,}")

Encoder loaded: 14 layers. Trainable parameters: 28,395,364


## Evaluation and thresholds
Checkpoint selection uses dev Macro-F1 with a tuned global threshold. Final reporting includes fixed 0.50, global, and per-statute thresholds. Predictions contain at least one statute, matching the existing Phase 5 pipeline. Dev-tuned scores are model-selection estimates.

In [7]:
def decisions(probabilities, threshold):
    predicted = probabilities >= threshold
    empty = ~predicted.any(axis=1)
    predicted[np.where(empty)[0], probabilities[empty].argmax(axis=1)] = True
    return predicted

def scores(labels, predicted):
    return {"macro_f1": float(f1_score(labels, predicted, average="macro", zero_division=0)),
            "micro_f1": float(f1_score(labels, predicted, average="micro", zero_division=0)),
            "precision": float(precision_score(labels, predicted, average="macro", zero_division=0)),
            "recall": float(recall_score(labels, predicted, average="macro", zero_division=0)),
            "subset_accuracy": float((labels == predicted).all(axis=1).mean())}

def calibrate(probabilities, labels, per_label=True):
    grid = np.linspace(0.02, 0.80, 40)
    best = max(grid, key=lambda t: f1_score(
        labels, decisions(probabilities, t), average="macro", zero_division=0))
    thresholds = np.full(N_LABELS, best, dtype=np.float32)
    if per_label:
        for j in range(N_LABELS):
            if labels[:, j].sum() >= 8:
                thresholds[j] = max(grid, key=lambda t: f1_score(
                    labels[:, j], probabilities[:, j] >= t, zero_division=0))
    return float(best), thresholds

@torch.inference_mode()
def predict(loader, label):
    model.eval()
    output = np.empty((len(loader.dataset), N_LABELS), dtype=np.float32)
    for tokens, lengths, owners, labels, indices in tqdm(loader, desc=label, unit="batch"):
        with autocast_context():
            probabilities = model(tokens, lengths, owners, len(labels))
        output[indices.numpy()] = probabilities.float().cpu().numpy()
    return output

def atomic_save(payload, path):
    temporary = path.with_suffix(".tmp.pt")
    torch.save(payload, temporary)
    os.replace(temporary, path)

run_config = {"chunk_len": CHUNK_LEN, "max_chunks": MAX_CHUNKS,
              "train_top_layers": TRAIN_TOP_LAYERS, "seed": SEED,
              "epochs": EPOCHS, "doc_batch": DOC_BATCH_SIZE, "grad_accum": GRAD_ACCUM,
              "backbone_lr": BACKBONE_LR, "head_lr": HEAD_LR,
              "train_key": train_data["key"], "dev_key": dev_data["key"],
              "checkpoint_hash": sha256_file(CHECKPOINT)}

## Train on every training document
Each epoch includes the complete training split. Progress bars show document loss and completed batches. Checkpoints resume from the last completed epoch. Increase EPOCHS in a fresh output directory when designing a longer experiment.

In [8]:
backbone_params = [p for name, p in model.named_parameters()
                   if p.requires_grad and not name.startswith("head.")]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_updates = EPOCHS * updates_per_epoch
warmup = max(1, round(total_updates * 0.06))
def schedule(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_updates - warmup))
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
scaler = torch.amp.GradScaler("cuda", enabled=AMP and AMP_DTYPE == torch.float16)
start_epoch, best_score, history = 0, -1.0, []
last_path, best_path = OUT_DIR / "last.pt", OUT_DIR / "best.pt"
if RESUME and last_path.exists():
    saved = torch.load(last_path, map_location="cpu", weights_only=False)
    if saved["run_config"] != run_config:
        raise ValueError("Settings changed. Choose a fresh OUT_DIR for this experiment.")
    model.load_state_dict(saved["model"])
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    scaler.load_state_dict(saved["scaler"])
    start_epoch, best_score, history = saved["epoch"], saved["best_score"], saved["history"]
    torch.set_rng_state(saved["torch_rng"])
    if AMP and saved["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(saved["cuda_rng"])
    del saved
    print(f"Resuming after epoch {start_epoch}.")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_sampler.epoch = epoch
    optimizer.zero_grad(set_to_none=True)
    running_loss, seen = 0.0, 0
    progress = tqdm(train_loader, desc=f"Train {epoch + 1}/{EPOCHS}", unit="batch")
    group_documents = 0
    for step, (tokens, lengths, owners, labels, indices) in enumerate(progress):
        count = len(labels)
        targets = labels.to(DEVICE, non_blocking=True)
        with autocast_context():
            probabilities = model(tokens, lengths, owners, count)
        loss = F.binary_cross_entropy(probabilities.clamp(1e-7, 1 - 1e-7), targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite document loss. Resume the last completed epoch after checking inputs.")
        scaler.scale(loss * count).backward()
        group_documents += count
        running_loss += loss.item() * count
        seen += count
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader):
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.div_(group_documents)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            group_documents = 0
        progress.set_postfix(loss=f"{running_loss / seen:.4f}", docs=seen)
    dev_prob = predict(dev_loader, f"Dev {epoch + 1}")
    threshold, _ = calibrate(dev_prob, dev_data["labels"], per_label=False)
    metrics = scores(dev_data["labels"], decisions(dev_prob, threshold))
    row = {"epoch": epoch + 1, "train_loss": running_loss / seen,
           "global_threshold": threshold, **metrics}
    history.append(row)
    print(f"Epoch {epoch + 1}: dev Macro-F1 {metrics['macro_f1']:.4f}, "
          f"Micro-F1 {metrics['micro_f1']:.4f}, threshold {threshold:.2f}")
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    if metrics["macro_f1"] > best_score:
        best_score = metrics["macro_f1"]
        atomic_save({"model": cpu_state, "backbone": vars(bc), "run_config": run_config,
                     "epoch": epoch + 1, "metrics": metrics}, best_path)
        np.save(OUT_DIR / "best_dev_probabilities.npy", dev_prob)
    atomic_save({"model": cpu_state, "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                 "epoch": epoch + 1, "best_score": best_score, "history": history,
                 "run_config": run_config, "torch_rng": torch.get_rng_state(),
                 "cuda_rng": torch.cuda.get_rng_state_all() if AMP else None}, last_path)
    (OUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
    del cpu_state
print("Training complete.")

Train 1/3:   0%|          | 0/21375 [00:00<?, ?batch/s]

Dev 1:   0%|          | 0/5091 [00:00<?, ?batch/s]

Epoch 1: dev Macro-F1 0.3034, Micro-F1 0.4059, threshold 0.10


Train 2/3:   0%|          | 0/21375 [00:00<?, ?batch/s]

Dev 2:   0%|          | 0/5091 [00:00<?, ?batch/s]

Epoch 2: dev Macro-F1 0.3507, Micro-F1 0.4769, threshold 0.16


Train 3/3:   0%|          | 0/21375 [00:00<?, ?batch/s]

Dev 3:   0%|          | 0/5091 [00:00<?, ?batch/s]

Epoch 3: dev Macro-F1 0.3679, Micro-F1 0.4896, threshold 0.16
Training complete.


## Final evaluation and test submission
The best checkpoint is evaluated with mean pooling. Thresholds selected on dev are saved and applied directly to test. The output has one record per official test ID.

In [9]:
best = torch.load(best_path, map_location="cpu", weights_only=False)
if best["run_config"] != run_config:
    raise ValueError("Best checkpoint belongs to a different experiment.")
model.load_state_dict(best["model"])
best_epoch = best["epoch"]
del best
prob = predict(dev_loader, "Final dev")
global_threshold, per_label_thresholds = calibrate(prob, dev_data["labels"])
reports = {
    "fixed_0.50": scores(dev_data["labels"], decisions(prob, 0.5)),
    "global": scores(dev_data["labels"], decisions(prob, global_threshold)),
    "per_label": scores(dev_data["labels"], decisions(prob, per_label_thresholds))}
for name, result in reports.items():
    print(f"{name:12s} Macro-F1 {result['macro_f1']:.4f} | "
          f"Micro-F1 {result['micro_f1']:.4f} | exact-set accuracy {result['subset_accuracy']:.4f}")
chosen = max(reports, key=lambda name: reports[name]["macro_f1"])
threshold = {"fixed_0.50": 0.5, "global": global_threshold,
             "per_label": per_label_thresholds}[chosen]
np.savez(OUT_DIR / "dev_predictions.npz", probabilities=prob, labels=dev_data["labels"],
         ids=np.asarray(dev_data["ids"]))
threshold_record = {"strategy": chosen, "threshold": np.asarray(threshold).tolist(),
                    "global": global_threshold, "per_label": per_label_thresholds.tolist()}
(OUT_DIR / "thresholds.json").write_text(json.dumps(threshold_record, indent=2))
(OUT_DIR / "results.json").write_text(json.dumps({
    "best_epoch": best_epoch, "representation": "up to 8 chunks, mean probabilities",
    "reports": reports, "run_config": run_config}, indent=2))
print(f"Selected threshold strategy: {chosen}")

if WRITE_SUBMISSION:
    test_data = prepare_split("test")
    test_prob = predict(evaluation_loader(test_data), "Test")
    predicted = decisions(test_prob, threshold)
    submission = [{"id": case_id, "pred": np.flatnonzero(row).tolist()}
                  for case_id, row in zip(test_data["ids"], predicted)]
    assert len(submission) == len(test_data["ids"])
    assert len({row["id"] for row in submission}) == len(submission)
    assert all(row["pred"] and len(row["pred"]) == len(set(row["pred"])) for row in submission)
    prediction_path = OUT_DIR / "prediction.json"
    prediction_path.write_text(json.dumps(submission))
    with zipfile.ZipFile(OUT_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    print(f"Saved {len(submission):,} test predictions to {prediction_path}")
    print(f"Submission archive: {OUT_DIR / 'submission.zip'}")

Final dev:   0%|          | 0/5091 [00:00<?, ?batch/s]

fixed_0.50   Macro-F1 0.2567 | Micro-F1 0.4241 | exact-set accuracy 0.0819
global       Macro-F1 0.3679 | Micro-F1 0.4896 | exact-set accuracy 0.0293
per_label    Macro-F1 0.3941 | Micro-F1 0.4897 | exact-set accuracy 0.0425
Selected threshold strategy: per_label


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing test:   0%|          | 0/13019 [00:00<?, ?document/s]

test: 13,019 documents, 2.77 chunks/document


Test:   0%|          | 0/6510 [00:00<?, ?batch/s]

Saved 13,019 test predictions to /content/drive/MyDrive/phase5_train_8chunks_mean/prediction.json
Submission archive: /content/drive/MyDrive/phase5_train_8chunks_mean/submission.zip
